# Data Preprocessing

In [1]:
import pandas as pd
import numpy as np
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df.head(3)

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.8,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.6,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.9,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.8,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.0,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.5,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN


In [4]:
df = df.drop(columns=['id', 'Unnamed: 32'])

In [7]:
from sklearn.model_selection import train_test_split
X = df.drop(columns=['diagnosis'])
y = df['diagnosis']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [8]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

In [9]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

In [10]:
# Convert numpy arrays to PyTorch tensors
import torch
X_train = torch.from_numpy(X_train)
X_test = torch.from_numpy(X_test)
y_train = torch.from_numpy(y_train)
y_test = torch.from_numpy(y_test)

# Defining the model

In [11]:
class Neural_Network():
  def __init__(self, X):
    self.weights = torch.rand(X.shape[1], 1, dtype=torch.float64, requires_grad=True)
    self.bias = torch.zeros(1, dtype=torch.float64, requires_grad=True)

  def forward(self, X):
    z = torch.matmul(X, self.weights) + self.bias
    y_pred = torch.sigmoid(z)
    return y_pred

  def loss_func(self, y_pred, y_train):
    epsilon = 1e-7
    y_pred = torch.clamp(y_pred, epsilon, 1 - epsilon)
    loss = -(y_train * torch.log(y_pred) + (1 - y_train) * torch.log(1 - y_pred)).mean()
    return loss

In [33]:
learning_rate = 0.1
epochs = 25

In [34]:
model = Neural_Network(X_train)

for epoch in range(epochs):
  # forward pass
  y_pred = model.forward(X_train)

  # loss calculate
  loss = model.loss_func(y_pred, y_train)

  # backward pass
  loss.backward()

  # parameters update
  with torch.no_grad():
    model.weights -= learning_rate*model.weights.grad
    model.bias -= learning_rate*model.bias.grad

  # zero gradients
  model.weights.grad.zero_()
  model.bias.grad.zero_()

  # Display loss in each epoch
  print(f'Epoch: {epoch+1}, Loss:{loss.item()}')

Epoch: 1, Loss:3.391298690905693
Epoch: 2, Loss:3.253943958647751
Epoch: 3, Loss:3.1088487484367513
Epoch: 4, Loss:2.9561023370931396
Epoch: 5, Loss:2.799180994618924
Epoch: 6, Loss:2.6426528408109875
Epoch: 7, Loss:2.4842063147412516
Epoch: 8, Loss:2.3238412241828854
Epoch: 9, Loss:2.1652100286261198
Epoch: 10, Loss:2.0089679270782015
Epoch: 11, Loss:1.850803217290086
Epoch: 12, Loss:1.7016695519352008
Epoch: 13, Loss:1.5615950384189037
Epoch: 14, Loss:1.4302655217643134
Epoch: 15, Loss:1.31269062377732
Epoch: 16, Loss:1.21021528692973
Epoch: 17, Loss:1.1233858050901246
Epoch: 18, Loss:1.0515988583303908
Epoch: 19, Loss:0.9934048148334057
Epoch: 20, Loss:0.946918646236224
Epoch: 21, Loss:0.910076579409198
Epoch: 22, Loss:0.8808562998517618
Epoch: 23, Loss:0.8574630398914739
Epoch: 24, Loss:0.8384350162356923
Epoch: 25, Loss:0.822659891426695


In [35]:
print(model.weights)

tensor([[-0.1666],
        [-0.1493],
        [-0.0500],
        [ 0.2416],
        [ 0.2078],
        [-0.4685],
        [ 0.0967],
        [-0.4497],
        [-0.2130],
        [ 0.2402],
        [ 0.0023],
        [ 0.4645],
        [-0.0649],
        [-0.2641],
        [ 0.6256],
        [ 0.2592],
        [-0.1527],
        [-0.2712],
        [ 0.0102],
        [ 0.0734],
        [-0.0693],
        [ 0.4613],
        [ 0.4727],
        [ 0.2923],
        [ 0.1503],
        [ 0.3257],
        [ 0.1930],
        [-0.2270],
        [ 0.4348],
        [-0.2500]], dtype=torch.float64, requires_grad=True)


In [36]:
print(model.bias)

tensor([-0.1812], dtype=torch.float64, requires_grad=True)


In [37]:
# Evaluation
with torch.no_grad():
  y_pred = model.forward(X_test)
  y_pred = (y_pred > 0.9).float()
  accuracy = (y_pred == y_test).float().mean()
  print(f'Accuracy: {accuracy}')

Accuracy: 0.6015697121620178
